# Pesquisa Comercial Dealwise (Python coleta, Gemini estrutura)

Para cada empresa da planilha, o notebook:
1. lê o site da empresa (página inicial e algumas páginas internas, como sobre, carreiras e clientes);
2. busca notícias, vagas, eventos e informações do lead na web;
3. envia esse material ao Gemini, que apenas organiza o texto nos 5 tópicos;
4. grava o resultado na coluna **Pesquisa IA** da própria planilha.

As empresas são processadas **uma por vez**. Se uma API key der erro (limite por minuto, limite diário ou key inválida), o notebook passa para a próxima key automaticamente.

**Como usar**
1. Rode a célula 1 e autorize o acesso ao Google (a conta precisa poder editar a planilha).
2. Na célula 2, cole as suas API keys.
3. Na planilha, abra a **aba que quer processar** e copie a URL do navegador (ela termina com `gid=...`, que identifica a aba).
4. Rode as células 3 e 4. A célula 4 pergunta a URL: cole a que você copiou. O notebook processa **todas as empresas pendentes daquela aba, uma por uma**.
5. Para outra aba, abra-a na planilha, copie a nova URL e rode a célula 4 de novo.

O Colab não enxerga qual aba está aberta no seu navegador. Por isso a URL com o `gid` é o jeito de indicar a aba.

Para um teste rápido, coloque `LIMITE_POR_ABA = 2`: ele processa só 2 empresas e mostra o resultado na tela.

Se a execução parar (limite das APIs, queda de conexão), rode a célula 4 de novo: as empresas já pesquisadas são puladas e ele continua de onde parou. Mantenha a aba do Colab aberta durante a execução.

Se quiser ver o que o código coletou de uma empresa antes de gastar uma chamada do Gemini, use a célula opcional 5.

**Atenção:** não compartilhe este notebook com as API keys coladas no código. Alternativa: guarde as keys em Secrets com os nomes `GEMINI_API_KEY_1` a `GEMINI_API_KEY_5` e deixe a lista vazia.

## 1. Instalação e autenticação

In [1]:
!pip -q install ddgs

import re, time, json, datetime, unicodedata, requests
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup

import gspread
from google.colab import auth, userdata
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print("Google autenticado.")

Google autenticado.


## 2. Configuração

In [ ]:
URL_PLANILHA = "https://docs.google.com/spreadsheets/d/19zrNQPO9ib6Ur9GPc0c1xSFq_ndWxl14jWEdt6CvQ3M/edit?gid=152758535#gid=152758535"  #@param {type:"string"}

# Nomes das abas separados por ponto e vírgula (opcional). Vazio = processa a aba do link.
ABAS = ""  #@param {type:"string"}

# True = com ABAS vazio, processa só a aba do link (a que tem "gid=" na URL).
# False = processa todas as abas que tiverem as colunas certas.
SOMENTE_ABA_DO_LINK = True  #@param {type:"boolean"}

# 0 = processa todas as empresas pendentes da aba, uma por uma. Use 2 ou 3 só para testar.
LIMITE_POR_ABA = 0  #@param {type:"integer"}

# True = ao rodar, pergunta a URL da aba que você quer processar (Enter = usa a URL acima).
PERGUNTAR_URL = False  #@param {type:"boolean"}

# True = além do site, busca notícias/vagas/eventos na web (biblioteca ddgs, não oficial).
USAR_BUSCA_WEB = True  #@param {type:"boolean"}

# Vazio = escolhe automaticamente o modelo "flash-lite" mais recente disponível para a sua key.
# Para fixar um modelo, escreva o ID aqui (ex.: um modelo flash-lite).
MODELO = ""  #@param {type:"string"}

# Cole aqui as keys dos seus projetos (as vazias são ignoradas).
API_KEYS = [
    "",
]

# ---- Ajustes avançados ----
LINHA_CABECALHO = 1
MAX_CHARS_CELULA = 6000
PAUSA_ENTRE_EMPRESAS_S = 2
MAX_TENTATIVAS_GEMINI = 12   # tentativas por empresa, somando todas as keys
MAX_PAGINAS_INTERNAS = 3     # páginas do site além da inicial
MAX_CHARS_PAGINA = 2000
MAX_RESULTADOS_BUSCA = 5
MAX_CHARS_MATERIAL = 16000
TIMEOUT_WEB_S = 15

COLUMNS = {
    "NOME_EMPRESA": "nome da empresa",
    "SITE_EMPRESA": "site da empresa",
    "LINKEDIN_PESSOA": "linkedin da pessoa",
    "NOME_PESSOA": "nome da pessoa",
    "SOBRENOME_PESSOA": "sobrenome da pessoa",
    "CARGO_PESSOA": "cargo da pessoa",
    "SETOR_EMPRESA": "setor da empresa",
    "PESQUISA": "Pesquisa IA",
}
OBRIGATORIAS = ["NOME_EMPRESA", "SITE_EMPRESA", "LINKEDIN_PESSOA"]

## 3. Funções

In [15]:
class ParadaTotal(Exception):
    pass

class CotaEsgotada(ParadaTotal):
    pass

class ModeloInvalido(ParadaTotal):
    pass


# ---------------- utilitários ----------------

def norm(t):
    s = unicodedata.normalize("NFD", str(t or ""))
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return re.sub(r"\s+", " ", s).strip().lower()


def limpar_texto(t):
    s = str(t or "")
    s = s.replace("**", "")
    s = re.sub(r"^#+\s*", "", s, flags=re.M)
    s = re.sub(r"^\s*[\*\-]\s+", "- ", s, flags=re.M)
    s = s.strip()
    if len(s) > MAX_CHARS_CELULA:
        s = s[: MAX_CHARS_CELULA - 3] + "..."
    return s


# ---------------- rodízio de API keys ----------------

class ChavesGemini:
    """Fica na mesma key até ela dar erro; então passa para a próxima."""

    def __init__(self, chaves):
        self.chaves = []
        for c in chaves:
            c = (c or "").strip()
            if c and c not in self.chaves:
                self.chaves.append(c)
        if not self.chaves:
            raise ValueError("Nenhuma API key informada. Preencha API_KEYS ou crie os Secrets GEMINI_API_KEY_1..5.")
        self.livre_em = {c: 0.0 for c in self.chaves}
        self.mortas = {}
        self.i = 0

    def nome(self, c):
        return f"#{self.chaves.index(c) + 1}"

    def avancar(self):
        self.i = (self.i + 1) % len(self.chaves)

    def proxima(self):
        while True:
            vivas = [c for c in self.chaves if c not in self.mortas]
            if not vivas:
                motivos = "; ".join(f"key {self.nome(c)}: {m}" for c, m in self.mortas.items())
                raise CotaEsgotada("Todas as API keys estão esgotadas ou inválidas. " + motivos)
            agora = time.time()
            n = len(self.chaves)
            for k in range(n):
                c = self.chaves[(self.i + k) % n]
                if c in self.mortas:
                    continue
                if self.livre_em[c] <= agora:
                    self.i = (self.i + k) % n
                    return c
            espera = min(self.livre_em[c] for c in vivas) - agora
            print(f"  Todas as keys em espera. Aguardando {max(espera, 1):.0f}s...")
            time.sleep(max(espera, 1))

    def esperar(self, c, segundos):
        self.livre_em[c] = time.time() + segundos
        self.avancar()

    def matar(self, c, motivo):
        self.mortas[c] = motivo[:200]
        self.avancar()


# ---------------- Gemini ----------------

def eh_cota_diaria(body):
    return bool(re.search(r"PerDay|per day|limit:\s*0", body, re.I))


def extrair_retry_delay(body):
    m = re.search(r'"retryDelay"\s*:\s*"(\d+(?:\.\d+)?)s"', body)
    return float(m.group(1)) if m else 0.0


def resumir_erro(body):
    try:
        j = json.loads(body)
        msg = (j.get("error") or {}).get("message", "")
        ids = " ".join(re.findall(r'"quotaId"\s*:\s*"[^"]+"', body))
        return f"{msg} {ids}".strip()[:400]
    except Exception:
        return str(body)[:400]


def escolher_modelo(chave):
    r = requests.get("https://generativelanguage.googleapis.com/v1beta/models?pageSize=200",
                     headers={"x-goog-api-key": chave}, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f"HTTP {r.status_code}: {resumir_erro(r.text)}")
    nomes = [m["name"].replace("models/", "") for m in r.json().get("models", [])
             if "generateContent" in m.get("supportedGenerationMethods", [])]
    excluir = ("image", "tts", "audio", "live", "native", "embedding", "vision",
               "robotics", "computer", "deep-research")
    nomes = [n for n in nomes if not any(x in n for x in excluir)]

    def versao(n):
        return [float(x) for x in re.findall(r"\d+(?:\.\d+)?", n)]

    for trecho in ("flash-lite", "flash"):
        cand = [n for n in nomes if trecho in n]
        if cand:
            return max(cand, key=versao)
    raise RuntimeError("Nenhum modelo flash encontrado para esta key.")


def chamar_gemini(prompt):
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{MODELO_ATIVO}:generateContent"
    corpo = {
        "contents": [{"role": "user", "parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": 0.2},
    }

    for _ in range(MAX_TENTATIVAS_GEMINI):
        chave = chaves.proxima()
        try:
            r = requests.post(url, headers={"x-goog-api-key": chave, "Content-Type": "application/json"},
                              json=corpo, timeout=120)
        except requests.RequestException as e:
            print(f"  Erro de rede com a key {chaves.nome(chave)}: {e}")
            chaves.esperar(chave, 5)
            continue

        if r.status_code == 200:
            try:
                cands = r.json().get("candidates") or []
                parts = ((cands[0].get("content") or {}).get("parts") or []) if cands else []
                texto = "".join(p.get("text", "") for p in parts).strip()
            except Exception:
                texto = ""
            if texto:
                return texto
            raise RuntimeError("Resposta vazia ou bloqueada pela API.")

        body = r.text
        if r.status_code == 429:
            if eh_cota_diaria(body):
                print(f"  Key {chaves.nome(chave)}: cota diária esgotada. Trocando de key.")
                chaves.matar(chave, resumir_erro(body))
            else:
                espera = extrair_retry_delay(body)
                espera = espera + 1 if espera else 60
                print(f"  Key {chaves.nome(chave)}: limite por minuto. Trocando de key (em espera {espera:.0f}s).")
                chaves.esperar(chave, espera)
            continue

        if r.status_code in (500, 503):
            print(f"  Key {chaves.nome(chave)}: API sobrecarregada (HTTP {r.status_code}). Trocando de key.")
            chaves.esperar(chave, 10)
            continue

        if r.status_code in (400, 401, 403) and re.search(r"API key|API_KEY|PERMISSION|permission|not valid|denied", body, re.I):
            print(f"  Key {chaves.nome(chave)} inválida ou sem permissão. Descartada.")
            chaves.matar(chave, f"HTTP {r.status_code}: {resumir_erro(body)}")
            continue

        if r.status_code == 404:
            raise ModeloInvalido(f"Modelo '{MODELO_ATIVO}' não encontrado: {resumir_erro(body)}")

        raise RuntimeError(f"HTTP {r.status_code}: {resumir_erro(body)}")

    raise RuntimeError("Falha na API Gemini após várias tentativas.")


# ---------------- coleta: site ----------------

HEADERS_WEB = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/124.0 Safari/537.36",
    "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
}

GRUPOS_PAGINAS = [
    ["sobre", "quem-somos", "quem somos", "about", "institucional", "empresa"],
    ["carreira", "trabalhe", "vagas", "career", "jobs"],
    ["cliente", "case", "portfolio", "parceiro"],
    ["produto", "solucao", "solucoes", "servico", "plataforma"],
    ["noticia", "imprensa", "news", "press", "blog"],
]

FERRAMENTAS = {
    "HubSpot": ["hubspot", "hs-scripts"],
    "RD Station": ["rdstation", "rd station"],
    "Salesforce/Pardot": ["salesforce", "pardot"],
    "Pipedrive": ["pipedrive"],
    "Calendly": ["calendly"],
    "Drift": ["drift.com", "driftt"],
    "Intercom": ["intercom"],
    "Zendesk chat": ["zdassets", "zopim"],
    "Tawk.to": ["tawk.to"],
    "JivoChat": ["jivosite", "jivochat"],
    "WhatsApp": ["wa.me", "api.whatsapp", "whatsapp"],
    "Google Tag Manager": ["googletagmanager"],
    "Meta Pixel": ["fbevents"],
    "LinkedIn Insight": ["snap.licdn"],
    "Typeform": ["typeform"],
}


def normalizar_url(site):
    s = (site or "").strip()
    if not s:
        return ""
    if not re.match(r"^https?://", s, re.I):
        s = "https://" + s
    return s


def baixar(url):
    try:
        r = requests.get(url, headers=HEADERS_WEB, timeout=TIMEOUT_WEB_S, allow_redirects=True)
        if r.status_code != 200 or "text/html" not in r.headers.get("Content-Type", ""):
            return None
        if not r.encoding or r.encoding.lower() == "iso-8859-1":
            r.encoding = "utf-8"
        return r.text
    except requests.RequestException:
        return None


def texto_da_pagina(html):
    soup = BeautifulSoup(html, "html.parser")
    titulo = soup.title.get_text(" ", strip=True) if soup.title else ""
    meta = soup.find("meta", attrs={"name": "description"})
    descricao = (meta.get("content") or "").strip() if meta else ""
    for tag in soup(["script", "style", "noscript", "svg", "iframe"]):
        tag.decompose()
    corpo = re.sub(r"\s+", " ", soup.get_text(" ", strip=True))
    return " | ".join(x for x in [titulo, descricao, corpo] if x)


def escolher_paginas(html, base):
    soup = BeautifulSoup(html, "html.parser")
    dominio = urlparse(base).netloc.replace("www.", "")
    links = []
    for a in soup.find_all("a", href=True):
        href = urljoin(base, a["href"]).split("#")[0]
        p = urlparse(href)
        if p.scheme not in ("http", "https"):
            continue
        if not p.netloc.replace("www.", "").endswith(dominio):
            continue
        if href.rstrip("/") == base.rstrip("/") or href in [l for l, _ in links]:
            continue
        links.append((href, norm(href + " " + a.get_text(" ", strip=True))))

    escolhidas = []
    for grupo in GRUPOS_PAGINAS:
        for href, alvo in links:
            if href not in escolhidas and any(k in alvo for k in grupo):
                escolhidas.append(href)
                break
    return escolhidas[:MAX_PAGINAS_INTERNAS]


def detectar_sinais(html):
    h = html.lower()
    achadas = [nome for nome, chaves_ in FERRAMENTAS.items() if any(k in h for k in chaves_)]
    sinais = []
    if achadas:
        sinais.append("ferramentas detectadas: " + ", ".join(achadas))
    sinais.append("formulario de contato na pagina inicial: " + ("sim" if "<form" in h else "nao"))
    return "; ".join(sinais)


# ---------------- coleta: busca na web ----------------

_aviso_busca = {"mostrado": False}


def buscar_web(consulta, n=5, noticias=False):
    try:
        try:
            from ddgs import DDGS
        except ImportError:
            from duckduckgo_search import DDGS
        with DDGS() as d:
            if noticias:
                res = list(d.news(consulta, region="br-pt", timelimit="y", max_results=n))
            else:
                res = list(d.text(consulta, region="br-pt", timelimit="y", max_results=n))
    except Exception as e:
        if not _aviso_busca["mostrado"]:
            print(f"  (busca web indisponível: {str(e)[:120]}. Seguindo só com o site.)")
            _aviso_busca["mostrado"] = True
        return ""

    linhas = []
    for x in res:
        titulo = (x.get("title") or "").strip()
        corpo = re.sub(r"\s+", " ", (x.get("body") or "")).strip()[:250]
        if noticias:
            data = (x.get("date") or "")[:10]
            fonte = x.get("source") or ""
            linhas.append(f"- [{data}] {titulo} | {corpo} | {fonte}")
        else:
            linhas.append(f"- {titulo} | {corpo} | {x.get('href') or x.get('url') or ''}")
    return "\n".join(linhas)


def coletar_material(d):
    partes = []

    site = normalizar_url(d["site"])
    if site:
        html = baixar(site)
        if html:
            partes.append("SITE (pagina inicial): " + texto_da_pagina(html)[:MAX_CHARS_PAGINA])
            partes.append("SINAIS NO SITE: " + detectar_sinais(html))
            for url in escolher_paginas(html, site):
                h = baixar(url)
                if h:
                    partes.append(f"SITE ({url}): " + texto_da_pagina(h)[:MAX_CHARS_PAGINA])
        else:
            partes.append(f"SITE: nao foi possivel acessar {site}.")

    if USAR_BUSCA_WEB:
        emp = d["empresa"]
        blocos = [
            ("NOTICIAS (ultimos 12 meses)", f'"{emp}"', True),
            ("VAGAS, EXPANSAO E INVESTIMENTOS", f'"{emp}" vagas comercial OR expansao OR investimento OR aquisicao', False),
            ("EVENTOS, WEBINARS E PREMIOS", f'"{emp}" evento OR webinar OR premio OR certificacao', False),
        ]
        if d["nome"]:
            blocos.append(("BUSCA SOBRE O LEAD (pode incluir homonimos)", f'"{d["nome"]}" "{emp}"', False))
        for titulo, consulta, noticias in blocos:
            linhas = buscar_web(consulta, MAX_RESULTADOS_BUSCA, noticias)
            if linhas:
                partes.append(titulo + ":\n" + linhas)
            time.sleep(1)

    material = "\n\n".join(partes).strip()
    return material[:MAX_CHARS_MATERIAL] if material else "(nenhum material coletado)"


# ---------------- prompt ----------------

def montar_prompt(d, material):
    hoje = datetime.date.today().strftime("%d/%m/%Y")

    return f"""Voce e um estrategista de inteligencia comercial B2B da Dealwise. Sua missao e analisar o material coletado de um prospect para entregar ao SDR uma leitura afiada do negocio, como ele vende e onde pode haver gargalos comerciais.

Sobre a Dealwise (quem somos):
Empresa de revenue building que atua dentro da operacao diaria de empresas B2B para construir receita previsivel, unindo:
- Negocios: desenho de processos comerciais, recrutamento, selecao e gestao de times de vendas.
- Tecnologia: ferramentas proprietarias de gestao comercial, RH e IA.
- Educacao: treinamento pratico de vendedores e liderancas.

Data de referencia: {hoje}

Alvo da pesquisa:
- Empresa: {d['empresa']}
- Site: {d['site'] or 'nao informado'}
- Setor: {d.get('setor') or 'nao informado'}
- Contato alvo (apenas contexto): {d['nome'] or 'nao informado'} ({d['cargo'] or 'cargo nao informado'})

Material coletado (site e buscas na web):
{material}

Regras estritas:
1. Baseie-se EXCLUSIVAMENTE nos fatos do material acima. Nao invente dados, mas faca leitura comercial inteligente do que esta escrito.
2. Ignore qualquer dado que pareca ser de homonimos ou de outras empresas.
3. Corte linguagem de marketing do site (nao use palavras vazias como "solucoes inteligentes", "inovador" ou "sem complicacao"). Traduza tudo para fatos objetivos.
4. Maximo de 1.150 caracteres no total.
5. Use apenas texto simples (proibido negrito, asteriscos, topicos com hifen ou markdown).
6. Nao repita instrucoes nos titulos. Escreva apenas o titulo exato e o texto abaixo.

Formato obrigatorio de saida (use exatamente estes 4 titulos, separados por uma linha em branco):

1. Raio-X do negocio
[Descreva de forma seca e objetiva: o que a empresa realmente vende, para quem (B2B, B2C ou ambos), modelo de receita/operacao e diferenciacao pratica encontrada no texto.]

2. Dinamica comercial e canais
[Identifique como a empresa parece capturar e atender clientes hoje (ex: autoatendimento digital, formulario, WhatsApp, venda consultiva), alem de sinais de expansao, vagas ou mudancas recentes (com mes/ano, se houver). Se nao houver noticias, foque nos canais e na estrutura visivel no site.]

3. Complexidade e contexto de mercado
[Aponte fatores relevantes identificados no material: regulacao do setor, atuacao hibrida (ex: PJ e PF juntos), novos produtos, eventos ou mencoes na midia nos ultimos 12 meses. Se nao houver noticias, destaque as caracteristicas operacionais do modelo dela.]

4. Hipotese de dor e abordagem Dealwise
[Em 2 frases diretas, aponte um provavel gargalo comercial gerado pelo modelo de negocio acima e conecte com um pilar especifico da Dealwise (processos/recrutamento, tecnologia/IA ou treinamento) para o SDR abrir a conversa.]"""


# ---------------- planilha ----------------

def mapear_colunas(headers):
    col = {}
    usadas = set()
    # 1a passada: nome exato do cabeçalho
    for k, nome in COLUMNS.items():
        alvo = norm(nome)
        if alvo in headers:
            col[k] = headers.index(alvo) + 1
            usadas.add(col[k])
    # 2a passada: cabeçalho que contém o nome, só em colunas ainda livres
    for k, nome in COLUMNS.items():
        if k in col:
            continue
        alvo = norm(nome)
        pos = next((i for i, h in enumerate(headers) if h and alvo in h and (i + 1) not in usadas), -1)
        if pos != -1:
            col[k] = pos + 1
            usadas.add(pos + 1)
    return col


def gravar_celula(ws, linha, coluna, valor):
    a1 = gspread.utils.rowcol_to_a1(linha, coluna)
    for tentativa in range(1, 6):
        try:
            ws.batch_update([{"range": a1, "values": [[valor]]}], value_input_option="RAW")
            return
        except gspread.exceptions.APIError as e:
            codigo = getattr(e.response, "status_code", 0)
            if codigo not in (429, 500, 503) or tentativa == 5:
                raise
            print("  Limite de escrita do Google Sheets, aguardando 30s...")
            time.sleep(30)


def processar_aba(ws, total):
    valores = ws.get_all_values()
    if len(valores) < LINHA_CABECALHO:
        print("  Aba vazia. Pulando.")
        return "pulada"

    headers = [norm(h) for h in valores[LINHA_CABECALHO - 1]]
    col = mapear_colunas(headers)
    faltando = [COLUMNS[k] for k in OBRIGATORIAS if k not in col]
    if faltando:
        print(f"  Pulando: colunas não encontradas: {', '.join(faltando)}")
        print(f"  Cabeçalho lido: {' | '.join(h for h in headers if h)[:200]}")
        return "pulada"

    if "PESQUISA" not in col:
        ultima = max((i + 1 for i, h in enumerate(headers) if h), default=0)
        nova = ultima + 1
        if nova > ws.col_count:
            ws.add_cols(nova - ws.col_count)
        ws.update_cell(LINHA_CABECALHO, nova, COLUMNS["PESQUISA"])
        col["PESQUISA"] = nova
        print(f"  Coluna 'Pesquisa IA' criada na coluna {nova}.")

    def pegar(linha, k):
        i = col.get(k)
        return linha[i - 1].strip() if i and i - 1 < len(linha) else ""

    fila = []
    for num, linha in enumerate(valores[LINHA_CABECALHO:], start=LINHA_CABECALHO + 1):
        atual = pegar(linha, "PESQUISA")
        empresa = pegar(linha, "NOME_EMPRESA")
        pendente = (not atual) or atual == "Pesquisando..." or atual.startswith("Erro:")
        if empresa and pendente:
            fila.append({"row": num, "dados": {
                "empresa": empresa,
                "site": pegar(linha, "SITE_EMPRESA"),
                "linkedin": pegar(linha, "LINKEDIN_PESSOA"),
                "nome": (pegar(linha, "NOME_PESSOA") + " " + pegar(linha, "SOBRENOME_PESSOA")).strip(),
                "cargo": pegar(linha, "CARGO_PESSOA"),
                "setor": pegar(linha, "SETOR_EMPRESA"),
            }})

    if LIMITE_POR_ABA > 0:
        fila = fila[:LIMITE_POR_ABA]
    if not fila:
        print("  Nenhuma empresa pendente.")
        return "ok"

    print(f"  {len(fila)} empresa(s) para pesquisar.")
    for n, it in enumerate(fila, start=1):
        d = it["dados"]
        t0 = time.time()
        print(f"  [{n}/{len(fila)}] {d['empresa']}")
        try:
            material = coletar_material(d)
            texto = limpar_texto(chamar_gemini(montar_prompt(d, material)))
            gravar_celula(ws, it["row"], col["PESQUISA"], texto)
            total["feitas"] += 1
            print(f"     ok ({time.time() - t0:.0f}s)")
            if LIMITE_POR_ABA > 0:
                print("\n" + texto + "\n")
        except ParadaTotal:
            raise
        except Exception as e:
            total["erros"] += 1
            print(f"     erro: {e}")
            gravar_celula(ws, it["row"], col["PESQUISA"], f"Erro: {str(e)[:300]}")
        time.sleep(PAUSA_ENTRE_EMPRESAS_S)
    return "ok"


# ---------------- teste de coleta (sem chamar o Gemini) ----------------

def testar_coleta(empresa, site, nome="", cargo=""):
    d = {"empresa": empresa, "site": site, "linkedin": "", "nome": nome, "cargo": cargo, "setor": ""}
    print(coletar_material(d))

## 4. Executar

In [22]:
url = URL_PLANILHA.strip()
if PERGUNTAR_URL:
    digitado = input("Cole a URL da aba que quer processar (Enter = usar a URL da configuração): ").strip()
    if digitado:
        url = digitado
if not url or "COLE_AQUI" in url:
    raise ValueError("Informe a URL da planilha.")

# Keys: lista da célula 2 ou, se estiver vazia, Secrets GEMINI_API_KEY_1..5
lista_keys = [k for k in API_KEYS if k and k.strip()]
if not lista_keys:
    for i in range(1, 6):
        try:
            v = userdata.get(f"GEMINI_API_KEY_{i}")
            if v:
                lista_keys.append(v)
        except Exception:
            pass

chaves = ChavesGemini(lista_keys)
print(f"{len(chaves.chaves)} API key(s) carregada(s).")

MODELO_ATIVO = MODELO.strip() or None
if not MODELO_ATIVO:
    for c in chaves.chaves:
        try:
            MODELO_ATIVO = escolher_modelo(c)
            break
        except Exception as e:
            print(f"  Não consegui listar modelos com a key {chaves.nome(c)}: {e}")
    if not MODELO_ATIVO:
        raise RuntimeError("Não foi possível escolher um modelo. Preencha MODELO na célula de configuração.")
print("Modelo:", MODELO_ATIVO)

planilha = gc.open_by_url(url)
m_gid = re.search(r"[#&?]gid=(\d+)", url)
if not ABAS.strip() and SOMENTE_ABA_DO_LINK:
    if not m_gid:
        raise RuntimeError("A URL não tem o gid da aba. Abra a aba desejada na planilha, copie a URL do "
                           "navegador (ela termina com gid=...) e cole aqui. Ou preencha ABAS com o nome da aba.")
    abas = [w for w in planilha.worksheets() if str(w.id) == m_gid.group(1)]
    if not abas:
        raise RuntimeError("Não encontrei a aba do link (gid). Informe o nome da aba em ABAS.")
elif ABAS.strip():
    abas = []
    for nome in [n.strip() for n in ABAS.split(";") if n.strip()]:
        try:
            abas.append(planilha.worksheet(nome))
        except gspread.exceptions.WorksheetNotFound:
            print(f"Aba '{nome}' não encontrada. Abas existentes: {[w.title for w in planilha.worksheets()]}")
else:
    abas = planilha.worksheets()

total = {"feitas": 0, "erros": 0}
inicio = time.time()
parou = None

try:
    for ws in abas:
        print(f"\n=== Aba: {ws.title} ===")
        processar_aba(ws, total)
except ParadaTotal as e:
    parou = e

print(f"\nConcluído em {time.time() - inicio:.0f}s | Pesquisadas: {total['feitas']} | Erros: {total['erros']}")
if parou:
    print("\nEXECUÇÃO INTERROMPIDA:", parou)
    print("O que já foi pesquisado está salvo na planilha. Rode novamente para continuar de onde parou.")
elif LIMITE_POR_ABA > 0:
    print("\nModo de teste (LIMITE_POR_ABA > 0). Confira a planilha e depois coloque LIMITE_POR_ABA = 0 para processar tudo.")

3 API key(s) carregada(s).
Modelo: gemini-3.5-flash-lite

=== Aba: Marketing ===
  Coluna 'Pesquisa IA' criada na coluna 18.
  50 empresa(s) para pesquisar.
  [1/50] Ybrida Comunicação em Saúde
  Key #1: API sobrecarregada (HTTP 503). Trocando de key.
     ok (36s, material de 8940 caracteres)
  [2/50] LGA Comunicação
  Key #2: API sobrecarregada (HTTP 503). Trocando de key.
  Key #3: API sobrecarregada (HTTP 503). Trocando de key.
  Key #1: API sobrecarregada (HTTP 503). Trocando de key.
  Todas as keys em espera. Aguardando 8s...
     ok (47s, material de 4715 caracteres)
  [3/50] Gruv
  Key #2: API sobrecarregada (HTTP 503). Trocando de key.
  Key #3: API sobrecarregada (HTTP 503). Trocando de key.
     ok (56s, material de 14087 caracteres)
  [4/50] ESCALA COMUNICACAO & MARKETING LTDA
  Key #1: API sobrecarregada (HTTP 503). Trocando de key.
  Key #2: API sobrecarregada (HTTP 503). Trocando de key.
  Key #3: API sobrecarregada (HTTP 503). Trocando de key.
  Key #1: API sobrecarrega

     ok (35s, material de 12232 caracteres)
  [13/50] Casa 96
     ok (28s, material de 4801 caracteres)
  [14/50] Smartalk
     ok (27s, material de 8097 caracteres)
  [15/50] Você Ligado
     ok (33s, material de 6762 caracteres)
  [16/50] Agência Orbe
     ok (31s, material de 58 caracteres)
  [17/50] Ivox Marketing Direto LTDA
     ok (27s, material de 1449 caracteres)
  [18/50] Estrutura de Comunicação
     ok (30s, material de 12362 caracteres)
  [19/50] Mosaico Media
     ok (33s, material de 3347 caracteres)
  [20/50] Grupo Alpes
     ok (33s, material de 2119 caracteres)
  [21/50] Agência Foster/WPP
     ok (33s, material de 3429 caracteres)
  [22/50] SURFE DIGITAL LTDA
     ok (33s, material de 10792 caracteres)
  [23/50] Mave Company
     ok (21s, material de 5781 caracteres)
  [24/50] Marke | Agência de Marketing Digital
     ok (34s, material de 8124 caracteres)
  [25/50] Front360 Comunicação Total
     ok (37s, material de 1934 caracteres)
  [26/50] Allma Marketing
     o